In [ ]:
%run /Users/calvin.waldheim@gmail.com/lakebase_config

In [0]:
# Cell 1 - Install driver
%pip install psycopg2-binary

In [0]:
# Cell 2 - Connect and create schema
import psycopg2



conn = psycopg2.connect(CONN_STRING, password=TOKEN)
cur = conn.cursor()

# Enable pgvector
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Memories table
cur.execute("""
CREATE TABLE IF NOT EXISTS memories (
  id              UUID PRIMARY KEY DEFAULT gen_random_uuid(),
  project_id      TEXT NOT NULL,
  project_type    TEXT NOT NULL,
  memory_type     TEXT NOT NULL CHECK (memory_type IN ('episodic', 'semantic')),
  scope           TEXT NOT NULL CHECK (scope IN ('personal', 'organizational')),
  user_id         TEXT,
  domain          TEXT,
  rule            TEXT NOT NULL,
  context         TEXT NOT NULL,
  source_ref      TEXT,
  source_version  BIGINT,
  content_hash    TEXT NOT NULL,
  embedding       VECTOR(1024),
  quality_score   FLOAT,
  retrieval_count INT DEFAULT 0,
  created_at      TIMESTAMPTZ DEFAULT now(),
  updated_at      TIMESTAMPTZ DEFAULT now(),
  metadata        JSONB
);
""")

# Retrieval log
cur.execute("""
CREATE TABLE IF NOT EXISTS retrieval_log (
  id            UUID PRIMARY KEY DEFAULT gen_random_uuid(),
  memory_id     UUID REFERENCES memories(id),
  project_id    TEXT NOT NULL,
  user_id       TEXT,
  query_text    TEXT,
  score         FLOAT,
  retrieved_at  TIMESTAMPTZ DEFAULT now()
);
""")

# Source registry
cur.execute("""
CREATE TABLE IF NOT EXISTS source_registry (
  source_ref      TEXT PRIMARY KEY,
  project_id      TEXT NOT NULL,
  source_type     TEXT,
  last_version    BIGINT,
  last_checked_at TIMESTAMPTZ
);
""")

# Indexes
cur.execute("CREATE INDEX IF NOT EXISTS memories_project_idx ON memories (project_id, memory_type, scope);")
cur.execute("CREATE INDEX IF NOT EXISTS memories_fts_idx ON memories USING gin (to_tsvector('english', rule || ' ' || context));")

conn.commit()
cur.close()
conn.close()
print("Schema created successfully.")